In [ ]:
import pandas as pd
import os
import functions_stanza
import multiprocessing
import pickle

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
dfs = []

for file in os.listdir("ingforms_download"):
    csv = pd.read_csv(f"ingforms_download/{file}", sep=";", header=None)
    csv = csv.set_index(csv[0])
    csv = csv.drop(columns=[0])
    dfs.append(csv)

In [ ]:
df = pd.concat(dfs)

In [ ]:
df = df.sort_index()

In [ ]:
df = df.drop_duplicates().reset_index(drop=True) # remove duplicates; export from kontext pretty iffy

In [ ]:
df = df.fillna("")

In [ ]:
df["english"] = df[2] + " " + df[3] + " " + df[4]

In [ ]:
replacements = [
    ("‘ ", "‘"),
    (" ’", "’"),
    (" ,", ","),
    (" .", "."),
    (" :", ":"),
    (" !", "!"),
    (" ?", "?"),
    ("( ", "("),
    (" )", ")"),
    (" ;", ";"),
    ("' ", "'"),
    (" '", "'"),
    (" -", "-"),
    ("- ", "-"),
    ("ş", "ș"),
    ("ţ", "ț"),
    ("Ş", "Ș"),
    ("Ţ", "Ț")
]

In [ ]:
for replacement in replacements:
    df["english"] = df["english"].str.replace(replacement[0], replacement[1])

In [ ]:
for replacement in replacements:
    df[7] = df[7].str.replace(replacement[0], replacement[1])

In [ ]:
df["romanian_new_orth"] = df[7].apply(
    functions_stanza.convert_sentence_to_new_orthography
)

In [ ]:
CHUNK_SIZE = 2500

# Parse English

In [ ]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_en = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks, args=(df, "en", "english", "ingforms", CHUNK_SIZE)
)

In [ ]:
process_en.start()

In [ ]:
multiprocessing.active_children()

In [ ]:
parsed_dict_en = functions_stanza.load_pickled_data(folder="ingforms", language="en")

In [ ]:
parsed_dict_en = pickle.load(open("ingforms/parsed_dict_en.pkl", "rb"))

In [ ]:
adv_part_indices_and_lemmas = []

In [ ]:
number = 1

for i in range(0, len(df)):
    row = df.iloc[i]
    contains_adv_part, lemma_of_adv_part = functions_stanza.sentences_contain_adv_part(
        row[3], parsed_dict_en[i]
    )
    if contains_adv_part:
        adv_part_indices_and_lemmas.append((i, lemma_of_adv_part))

In [ ]:
adv_part_indices_and_lemmas

In [ ]:
len(adv_part_indices_and_lemmas)

# Parse Romanian

In [ ]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_ro = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks,
    args=(df, "ro", "romanian_new_orth", "ingforms", CHUNK_SIZE),
)

In [ ]:
process_ro.start()

In [ ]:
multiprocessing.active_children()

In [ ]:
parsed_dict_ro = functions_stanza.load_pickled_data(folder="ingforms", language="ro")

In [ ]:
parsed_dict_ro = pickle.load(open("ingforms/parsed_dict_ro.pkl", "rb"))

In [ ]:
gerunziu_indices_and_words = []

In [ ]:
for i in range(0, len(df)):
    gerunzius = functions_stanza.get_gerunzius_in_sentences(parsed_dict_ro[i])
    if gerunzius:
        gerunziu_indices_and_words.append((i, gerunzius))

# Building combined df

In [ ]:
adv_part_indices_and_lemmas_df = pd.DataFrame(
    adv_part_indices_and_lemmas, columns=["index", "lemma"]
).set_index("index")

In [ ]:
gerunziu_indices_and_words_df = pd.DataFrame(
    gerunziu_indices_and_words, columns=["index", "gerunzius_and_lemmas"]
).set_index("index")

In [ ]:
df = pd.concat(
    [df, adv_part_indices_and_lemmas_df, gerunziu_indices_and_words_df], axis=1
)

In [ ]:
df["is_adv_participle"] = False
df.loc[~df["lemma"].isna(), "is_adv_participle"] = True

In [ ]:
df["has_gerunziu"] = False
df.loc[~df["gerunzius_and_lemmas"].isna(), "has_gerunziu"] = True

In [ ]:
df.loc[df["is_adv_participle"] & df["has_gerunziu"]].to_csv("extraction_output/adv_part_with_gerunziu.csv")

In [ ]:
df.to_csv("extraction_output/annotated_all_part_to_ger.csv")

# Assigning adverbial participles to gerunzius

In [ ]:
df = pd.read_csv("extraction_output/adv_part_with_gerunziu.csv", index_col=0)

In [ ]:
import functions_simalign

In [ ]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_matching = multiprocessing.Process(
    target=functions_simalign.match_adv_part_and_gerunzius, args=(df, "en")
)

In [ ]:
process_matching.start()

In [ ]:
multiprocessing.active_children()

In [ ]:
matching_df, errors = pickle.load(open("ingforms/matching.pkl", "rb"))

In [ ]:
matching_df.to_csv("extraction_output/adv_ger_matches.csv")